# Первичный аудит и предобработка данных (ДЗ 1)
**Кейс:** Исследование продаж и факторов, влияющих на динамику выручки в розничной торговле.  
**Цель ноутбука:** проверить качество исходных таблиц (`stores`, `features`, `sales`), выявить аномалии и расхождения с ТЗ, собрать единую витрину для дальнейшего EDA и проверки гипотез.

**План работы:**
1. Загрузка данных из локального хранилища / репозитория и проверка структуры.
2. Аудит качества: пропуски, дубликаты, форматы дат и выбросы.
3. Проверка ключей, объединение таблиц и генерация базовых признаков.
4. Фиксация выводов и вопросов к заказчику.

In [1]:
import os
import pandas as pd
import numpy as np

files = {
    "stores": "stores_data_set.csv",
    "features": "Features_data_set.csv",
    "sales": "sales_data_set.csv",
}

def load_data(filename):
    if os.path.exists(f"sample_data/{filename}"):
        return pd.read_csv(f"sample_data/{filename}")
    elif os.path.exists(f"../data/{filename}"):
        return pd.read_csv(f"../data/{filename}")
    else:
        url = f"https://githubusercontent.com{filename}"
        return pd.read_csv(url)

stores = load_data(files["stores"])
features = load_data(files["features"])
sales = load_data(files["sales"])


### 1. Проверка типов данных и приведение дат
При первичном осмотре видно, что даты в `features` и `sales` записаны в европейском формате `DD/MM/YYYY` (например, `19/02/2010`). При стандартном парсинге без указания формата день и месяц могут перепутаться, поэтому приводим их явно через `format='%d/%m/%Y'`.

In [2]:
features["Date"] = pd.to_datetime(features["Date"], format="%d/%m/%Y")
sales["Date"] = pd.to_datetime(sales["Date"], format="%d/%m/%Y")

print(f"Период в sales:    {sales['Date'].min():%d.%m.%Y} — {sales['Date'].max():%d.%m.%Y} ({sales['Date'].nunique()} нед.)")
print(f"Период в features: {features['Date'].min():%d.%m.%Y} — {features['Date'].max():%d.%m.%Y} ({features['Date'].nunique()} нед.)")
print(f"Уникальных магазинов: {sales['Store'].nunique()}, отделов: {sales['Dept'].nunique()}")

Период в sales:    05.02.2010 — 26.10.2012 (143 нед.)
Период в features: 05.02.2010 — 26.07.2013 (182 нед.)
Уникальных магазинов: 45, отделов: 81


### 2. Аудит пропусков и дубликатов
Проверим долю пропущенных значений (`NA`) и наличие дублей по бизнес-ключам:
* для `stores` — `Store`;
* для `features` — `(Store, Date)`;
* для `sales` — `(Store, Dept, Date)` (уровень наблюдения: магазин — отдел — неделя).

In [3]:
# Считаем пропуски только там, где они есть (в features)
na_summary = pd.DataFrame({
    "Пропусков (шт.)": features.isna().sum(),
    "Доля (%)": (features.isna().mean() * 100).round(2)
})
print("Пропуски в таблице features:")
display(na_summary[na_summary["Пропусков (шт.)"] > 0])

print(f"Пропуски в stores: {stores.isna().sum().sum()} | в sales: {sales.isna().sum().sum()}")

# Проверка дубликатов
dups = pd.Series({
    "stores (по Store)": stores.duplicated(subset=["Store"]).sum(),
    "features (по Store + Date)": features.duplicated(subset=["Store", "Date"]).sum(),
    "sales (по Store + Dept + Date)": sales.duplicated(subset=["Store", "Dept", "Date"]).sum(),
})
print("\nДубликаты по ключам:")
display(dups.to_frame("Количество дублей"))

Пропуски в таблице features:


,Пропусков (шт.),Доля (%)
MarkDown1,4158,50.77
MarkDown2,5269,64.33
MarkDown3,4577,55.89
MarkDown4,4726,57.70
MarkDown5,4140,50.55
CPI,585,7.14
Unemployment,585,7.14


Пропуски в stores: 0 | в sales: 0

Дубликаты по ключам:


,Количество дублей
stores (по Store),0
features (по Store + Date),0
sales (по Store + Dept + Date),0


### 3. Проверка аномалий и выбросов
Посмотрим на описательные статистики числовых полей. Особое внимание уделим отрицательным значениям в недельных продажах (`Weekly_Sales`) и промо-уценках (`MarkDown1–5`).

In [4]:
display(sales[["Weekly_Sales"]].describe().T)

neg_sales = sales[sales["Weekly_Sales"] < 0]
zero_sales = sales[sales["Weekly_Sales"] == 0]
print(f"Отрицательные продажи: {len(neg_sales):,} строк ({len(neg_sales)/len(sales):.2%}), минимум: {sales['Weekly_Sales'].min():.2f}")
print(f"Нулевые продажи: {len(zero_sales)} строк")

# Проверка отрицательных значений в скидках
md_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
neg_md = pd.DataFrame({
    "Отрицательных значений": (features[md_cols] < 0).sum(),
    "Минимум": features[md_cols].min()
})
print("\nАномалии в промо-акциях (MarkDown):")
display(neg_md)

,count,mean,std,min,25%,50%,75%,max
Weekly_Sales,421570.0,15981.258123,22711.183519,-4988.94,2079.65,7612.03,20205.8525,693099.36


Отрицательные продажи: 1,285 строк (0.30%), минимум: -4988.94
Нулевые продажи: 73 строк

Аномалии в промо-акциях (MarkDown):


,Отрицательных значений,Минимум
MarkDown1,4,-2781.45
MarkDown2,25,-265.76
MarkDown3,13,-179.26
MarkDown4,0,0.22
MarkDown5,2,-185.17


### 4. Объединение таблиц и подготовка признаков
Перед объединением проверим, совпадает ли признак `IsHoliday` в `sales` и `features`, чтобы не получить дубли столбцов. Затем присоединим справочники через `left join` к `sales` (тем самым отсечём «будущий» период 2012–2013 гг. из `features`, где нет данных о продажах и пропущены `CPI`/`Unemployment`).

In [5]:
# Проверка согласованности IsHoliday перед слиянием
holiday_check = (
    sales[["Store", "Date", "IsHoliday"]].drop_duplicates()
    .merge(features[["Store", "Date", "IsHoliday"]], on=["Store", "Date"], suffixes=("_s", "_f"))
)
assert (holiday_check["IsHoliday_s"] == holiday_check["IsHoliday_f"]).all(), "Есть расхождения в IsHoliday!"

# Сборка единой витрины
df = (
    sales
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(stores, on="Store", how="left")
)

# Календарные признаки
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

# Признаки промо-акций (учёт MarkDown начался с 11.11.2011)
df["Is_MarkDown_Period"] = (df["Date"] >= "2011-11-11").astype(int)
df[md_cols] = df[md_cols].fillna(0).clip(lower=0)
df["Total_MarkDown"] = df[md_cols].sum(axis=1)
df["Has_MarkDown"] = (df["Total_MarkDown"] > 0).astype(int)

# Температура в градусах Цельсия
df["Temp_C"] = ((df["Temperature"] - 32) * 5 / 9).round(1)

print(f"Итоговая витрина: {df.shape[0]:,} строк, {df.shape[1]} столбцов")
print(f"Осталось пропусков: {df.isna().sum().sum()}")
display(df.head(5))

Итоговая витрина: 421,570 строк, 23 столбцов
Осталось пропусков: 0


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Unemployment,Type,Size,Year,Month,Week,Is_MarkDown_Period,Total_MarkDown,Has_MarkDown,Temp_C
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,...,8.106,A,151315,2010,2,5,0,0.0,0,5.7
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,...,8.106,A,151315,2010,2,6,0,0.0,0,3.6
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,...,8.106,A,151315,2010,2,7,0,0.0,0,4.4
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,...,8.106,A,151315,2010,2,8,0,0.0,0,8.1
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,...,8.106,A,151315,2010,3,9,0,0.0,0,8.1


### 5. Итоги аудита и вопросы к заказчику

**Что выявлено и исправлено:**
* **Временной охват:** `features.csv` содержит 182 недели (по 26.07.2013), а `sales.csv` — 143 недели (по 26.10.2012). При `left join` к продажам период без целевой переменной корректно отсечён вместе с пропусками в `CPI` и `Unemployment` (которые начинаются с мая 2013 г.).
* **Формат дат и ключи:** европейский формат `DD/MM/YYYY` приведён к `datetime`, дублей по ключам нет, флаг `IsHoliday` полностью согласован между таблицами.
* **Промо-акции (`MarkDown1–5`):** до 11.11.2011 данные отсутствуют полностью (`NA`), после этой даты пропуски означают отсутствие акции (заменены на `0`).

**Вопросы и ограничения для обсуждения с заказчиком:**
1. **Противоречие в ТЗ:** в описании задач упомянута сегментация клиентов, но уровень гранулярности данных — «магазин — отдел — неделя» (Customer ID отсутствуют). Работаем в рамках анализа выручки и промо-факторов.
2. **Отсутствие региона:** в `stores.csv` нет колонки региона. Планируем проверить возможность косвенной группировки магазинов по одинаковым макропрофилям (`CPI` и `Unemployment`).
3. **Отрицательные значения:** в `Weekly_Sales` есть отрицательные значения (интерпретируем как возвраты товара и оставляем в расчётах чистой выручки), а в `MarkDown` встречаются редкие отрицательные значения (обнулены при расчёте суммарного промо-бюджета; уточнить у заказчика, являются ли они корректировками или ошибками выгрузки).